# Gridworld plots

Figures for the gridworld (DQN) experiments. Run `run.sh` first to populate
`results/`, then replace each `'TODO: env_name run'` placeholder below with the
name of an experiment directory you produced (e.g.
`mcvl_TomatoWateringEnvironment`). Figures are saved to `plots/`.


In [ ]:
from typing import Dict, Any, List

import numpy as np
%load_ext autoreload
%autoreload 2

In [ ]:
# from environment_utils import *  # not needed for plotting
from matplotlib import pyplot as plt
import matplotlib
from matplotlib.patches import Rectangle
import scipy.stats
from scipy.ndimage import uniform_filter1d
from scipy.stats import bootstrap


import os
import dataclasses
import json
# from mpl_sizes import get_format

# formatter = get_format("NeurIPS") # options: ICLR, ICML, NeurIPS, InfThesis

blue = '#4882a6'
green = '#5eae94'
orange = '#e15e45'
purple = '#6D247A'
pink = '#AB1368'
yellow = '#F1C500'
light_blue = '#7DAED9'
brown = '#7f4f24'  # Freeze RM baseline
grey = '#efefef'
legend_color = '#ffffff'

In [ ]:
def load_metrics(run, d):
    print('loading metrics for', run)
    initial_metrics = []
    tampering_metrics = []
    no_tampering_metrics = []
    has_tampering = True
    for seed in d['seeds']:
        with open(f'results/{run}/{seed}_initial_metrics.json', 'r') as f:
            initial_metrics.append(json.load(f))
        # The No-Gate baseline has no tampering deployment, so this file may be absent.
        tampering_path = f'results/{run}/{seed}_deployment_tampering_metrics.json'
        if os.path.exists(tampering_path):
            with open(tampering_path, 'r') as f:
                tampering_metrics.append(json.load(f))
        else:
            has_tampering = False
        with open(f'results/{run}/{seed}_deployment_no_tampering_metrics.json', 'r') as f:
            no_tampering_metrics.append(json.load(f))

    def dict_list_to_dict_of_lists(ld: list[dict]) -> dict[Any, list[Any]]:
        lists = {k: [dic[k] for dic in ld] for k in ld[0]}
        for k in ld[0]:
            try:
                lists[k] = np.array(lists[k])
            except ValueError:
                print(f'Could not convert {k} to np.array, keeping as list of lists')
        return lists
    
    initial_metrics = dict_list_to_dict_of_lists(initial_metrics)
    tampering_metrics = dict_list_to_dict_of_lists(tampering_metrics) if has_tampering else {}
    no_tampering_metrics = dict_list_to_dict_of_lists(no_tampering_metrics)
    return initial_metrics, tampering_metrics, no_tampering_metrics


In [ ]:
import numpy as np
import dataclasses


@dataclasses.dataclass
class RunInfo:
    run: str
    label: str
    color: str
    deployment_label: str | None = None
    deployment_color: str | None = None

@dataclasses.dataclass
class PlotConfig():
    x_min: float = 0
    x_max: float = 1
    y_min: float = 0
    y_max: float = 1
    show_legend: bool = False
    show_x_label: bool = False
    show_y_label: bool = False
    y_ticks: np.array = np.zeros(1)
    true_performance_y_min: float = None
    true_performance_y_max: float = None
    true_performance_y_ticks: np.array = None
    observed_return_y_min: float = None
    observed_return_y_max: float = None
    observed_return_y_ticks: np.array = None
    x_ticks: np.array = None
    env_name: str = 'NotDefined'
    show_divider: bool = False
    divider_label: str = 'Switch to $\mathit{Full}$'
    smoothing: int = 1
    legend_ncols: int = 4
    save_legend_separately: bool = False
    legend_offset_y: float = 0
    legend_offset_x: float = 0
    legend_columnspacing: float = 0
    override_with_env_defaults: bool = True
    vertical_layout: bool = False
    show_x_ticks: bool = True


def mean_confidence_interval(data, confidence=0.95):
    # compute mean and confidence interval using scipy.stats.bootstrap
    m = np.mean(data, axis=0)
    data = (data,)
    bootstrap_ci = bootstrap(data, 
                             statistic=np.mean,
                             n_resamples=1000, 
                             confidence_level=confidence,
                             method='percentile',
                             axis=0).confidence_interval
    return m, bootstrap_ci.low, bootstrap_ci.high

def smoothen(x, config):
    return uniform_filter1d(x, size=config.smoothing)

def resolve_y_axis_config(config, axis_name):
    if axis_name == 'true_performance':
        y_min = config.true_performance_y_min
        y_max = config.true_performance_y_max
        y_ticks = config.true_performance_y_ticks
    elif axis_name == 'observed_return':
        y_min = config.observed_return_y_min
        y_max = config.observed_return_y_max
        y_ticks = config.observed_return_y_ticks
    else:
        raise ValueError(f'Unknown axis name: {axis_name}')
    if y_min is None:
        y_min = config.y_min
    if y_max is None:
        y_max = config.y_max
    if y_ticks is None:
        y_ticks = config.y_ticks
    return y_min, y_max, y_ticks

def plot_line(ax, x, y, c, label, config):
    x = x[0]
    y_mean, y_cfm, y_cfp = mean_confidence_interval(y)
    # y_mean, y_cfm, y_cfp = y.mean(0), y.mean(0) - y.std(0),  y.mean(0) + y.std(0)
    y_mean, y_cfm, y_cfp = smoothen(y_mean, config), smoothen(y_cfm, config), smoothen(y_cfp, config)
    ax.plot(x, y_mean, c=c, label=label)
    ax.fill_between(x, y_cfm, y_cfp, color=c, alpha=.3)

def plot_metrics(axs, metrics, conf, init=None, col='r', label='', name=''):
    returns = metrics['eval_returns'].copy()
    performance = metrics['eval_performances'].copy()
    x = metrics['eval_steps'].copy()
    if init is not None and init.keys():
        x += init['eval_steps'][-1,-1]
        x = np.concatenate([init['eval_steps'][:, -1, None], x], axis=-1)
        returns = np.concatenate([init['eval_returns'][:, -1, None], returns], axis=-1)
        performance = np.concatenate([init['eval_performances'][:, -1, None], performance], axis=-1)
    elif conf.show_divider:
        for ax in axs:
            ax.axvline(x[-1,-1], linestyle='dashed', c='k', label=conf.divider_label, linewidth=1)
    plot_line(axs[1], x, returns, c=col, label=label, config=conf)
    plot_line(axs[0], x, performance, c=col, label=label, config=conf)
        
    for ax in axs:
        ax.set_xlim(conf.x_min, conf.x_max)
        ax.spines[['right', 'top', 'bottom']].set_visible(False)
        if conf.show_x_label:
            ax.set_xlabel('steps')
        ax.set_facecolor(grey)
        ax.grid(axis='y', color='white')
        if conf.x_ticks is not None:
            ax.set_xticks(conf.x_ticks)
    performance_y_min, performance_y_max, performance_y_ticks = resolve_y_axis_config(conf, 'true_performance')
    observed_return_y_min, observed_return_y_max, observed_return_y_ticks = resolve_y_axis_config(conf, 'observed_return')
    axs[0].set_ylim(performance_y_min, performance_y_max)
    if performance_y_ticks is not None:
        axs[0].set_yticks(performance_y_ticks)
    axs[1].set_ylim(observed_return_y_min, observed_return_y_max)
    if observed_return_y_ticks is not None:
        axs[1].set_yticks(observed_return_y_ticks)

    if not conf.show_x_ticks:
            axs[0].set_xticks([])
    if conf.vertical_layout:
        axs[0].set_xlabel(None)
    if conf.show_legend:
        handles, labels = ax.get_legend_handles_labels()
        def atoi(text):
            return int(text) if text.isdigit() else text
        # sort both labels and handles by labels
        labels, handles = zip(*sorted(zip(labels, handles), key=lambda t: [ atoi(c) for c in re.split(r'(\d+)', t[0]) ]))
        legend_offset = -0.72 + conf.legend_offset_y
        if conf.show_x_label:
            legend_offset += -0.22
        legend = axs[0].legend(loc='lower center', bbox_to_anchor=(1.05 + conf.legend_offset_x, legend_offset),
          fancybox=True, shadow=False, ncol=conf.legend_ncols, handles=handles, labelspacing=0, columnspacing=1.5 + conf.legend_columnspacing, facecolor=legend_color, borderpad=0.3, edgecolor=legend_color)

        def export_legend(legend, filename=f"plots/{name}_legend.pdf"):
            fig  = legend.figure
            fig.canvas.draw()
            bbox  = legend.get_window_extent()
            bbox = bbox.from_extents(*(bbox.extents))
            bbox = bbox.transformed(fig.dpi_scale_trans.inverted())
            fig.savefig(filename, bbox_inches=bbox)
        
        if conf.save_legend_separately:
            export_legend(legend)
            legend.remove()
    
    # axs[0].set_title(f'{env_name} Episode Return')
    # axs[1].set_title(f'{env_name} Episode Performance')
    if conf.show_y_label:
        axs[0].set_ylabel('true performance', rotation=0, loc='top', labelpad=-90, fontsize=10)
        axs[1].set_ylabel('observed return', rotation=0, loc='top', labelpad=-85, fontsize=10)

def plot_run(run, conf):
    plot_multirun([RunInfo(run, 'MC-DDQN (ours)', green,
                           deployment_label='DDQN',
                           deployment_color=orange,
                           )], conf, name=run)

def plot_run_oracle(run, oracle, conf, baseline=None):
    runs = [
        RunInfo(oracle, 'Oracle', yellow, deployment_label='Frozen', deployment_color=light_blue),
        RunInfo(run, 'MC-DDQN (ours)', green, deployment_label='DDQN', deployment_color=orange),
    ]
    if baseline is not None:
        # "No-Gate" / "Reward-Model RL" baseline (frozen R_psi, no gate): no
        # deployment_label, so only its no-tampering curve is drawn as the main line.
        runs.append(RunInfo(baseline, 'Freeze RM', brown))
    plot_multirun(runs, conf, name=(f'{run}_freezerm' if baseline is not None else run))
    
    
def plot_multirun(runs_infos: list[RunInfo], conf, name, tampering_label=None, tampering_color=orange):
    plt.rcParams["font.family"] = "Times New Roman"
    plt.rcParams["font.size"] = "10"
    plt.rcParams["font.serif"] = ["Times New Roman"]
    plt.rcParams['mathtext.fontset'] = 'custom'
    plt.rcParams['mathtext.rm'] = 'Times New Roman'
    plt.rcParams['mathtext.it'] = 'Times New Roman:italic'

    fig_height = 1.1
    if conf.show_y_label:
        fig_height += 0.1
    if conf.show_x_label:
        fig_height += 0.1

    if conf.vertical_layout:
        fig_height = 3.5
    if conf.vertical_layout:
        fig, axs = plt.subplots(2, 1, figsize=(1.5, fig_height))
    else:
        fig, axs = plt.subplots(1, 2, figsize=(3.5, fig_height))
    # fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    fig.subplots_adjust(wspace=0.4)
    # fig, axs = plt.subplots(1, 2, figsize=(7, 3))
    for i, run_info in enumerate(runs_infos):
        run = run_info.run
        if conf.override_with_env_defaults:
            change_config_for_env(run, conf)
        with open(f'results/{run}/config.json', 'r') as f:
            d = json.load(f)
            
        initial_metrics, tampering_metrics, no_tampering_metrics = load_metrics(run, d)
        if 'eval_steps' not in initial_metrics.keys():
            init_steps = 0
        else:
            init_steps = initial_metrics['eval_steps'].max()
        conf.x_max = init_steps + no_tampering_metrics['eval_steps'].max()+2
        if run_info.deployment_label is not None:
            plot_metrics(axs, tampering_metrics, conf, init=initial_metrics, col=run_info.deployment_color, label=run_info.deployment_label, name=name)
        if i == len(runs_infos) - 1:
            plot_metrics(axs, initial_metrics, conf, col=blue, label='Pretraining', name=name)

        plot_metrics(axs, no_tampering_metrics, conf, init=initial_metrics, col=run_info.color, label=run_info.label, name=name)
    fig.patch.set_facecolor('None')
    fig.patch.set_alpha(0)
    fig.savefig(f'plots/{name}.pdf', facecolor=fig.get_facecolor(), bbox_inches='tight')
    print('saved as\n', f'plots/{name}.pdf')
    plt.show()


def change_config_for_env(run, conf):
    if 'BoxMovingEnv' in run:
        conf.y_min = -59
        conf.y_max = 219
        conf.y_ticks = np.arange(-50, 250, 50)
        conf.env_name = 'Box Moving'
        conf.true_performance_y_min = -59
        conf.true_performance_y_max = 55
        conf.true_performance_y_ticks = np.arange(-50, 55, 25)
        conf.observed_return_y_min = -20
        conf.observed_return_y_max = 219
        conf.observed_return_y_ticks = np.arange(0, 250, 50)
    if 'AbsentSupervisorEnvironment' in run:
        conf.y_min = -100
        conf.y_max = 60
        conf.env_name = 'Absent Supervisor'
        conf.y_ticks = np.arange(-75, 75, 25)
    if 'TomatoWateringEnvironment' in run:
        conf.env_name = 'Tomato Watering'
        conf.true_performance_y_min = -1
        conf.true_performance_y_max = 16
        conf.true_performance_y_ticks = np.arange(0, 20, 5)
        conf.observed_return_y_min = -1
        conf.observed_return_y_max = 42
        conf.observed_return_y_ticks = np.arange(0, 45, 10)
    if 'RocksDiamondsEnvironment' in run:
        conf.y_min = -150
        conf.y_max = 1100
        conf.y_ticks = np.arange(0, 1200, 200)
        conf.x_ticks = np.arange(0, 30001, 15000)
        conf.env_name = 'Rocks'
        conf.true_performance_y_min = -105
        conf.true_performance_y_max = 105
        conf.true_performance_y_ticks = np.arange(-100, 105, 50)
        conf.observed_return_y_min = -10
        conf.observed_return_y_max = 1100
        conf.observed_return_y_ticks = np.arange(0, 1200, 200)


In [ ]:
# Main results: per-environment MCVL vs. Oracle.
# Replace each placeholder with the experiment directory produced under results/
# (e.g. 'mcvl_TomatoWateringEnvironment' and the matching oracle run). The run
# directory name must contain the environment class name, since
# change_config_for_env() applies the per-environment axis settings from it.

# Absent Supervisor
plot_run_oracle('TODO: env_name run', 'TODO: env_name run', PlotConfig(
    show_y_label=False,
    vertical_layout=True,
    show_x_ticks=False,
    show_x_label=True,
))
# Tomato Watering
plot_run_oracle('TODO: env_name run', 'TODO: env_name run', PlotConfig(
    show_x_label=True,
    vertical_layout=True,
    show_x_ticks=False,
))
# Rocks & Diamonds
plot_run_oracle('TODO: env_name run', 'TODO: env_name run', PlotConfig(
    show_x_label=True,
    vertical_layout=True,
    show_x_ticks=False,
))
# Box Moving
plot_run_oracle('TODO: env_name run', 'TODO: env_name run', PlotConfig(
    show_y_label=True,
    show_legend=False,
    save_legend_separately=True,
    vertical_layout=True,
    show_x_ticks=False,
    show_x_label=True,
))


## Ablation & appendix figures

These reproduce the appendix ablation studies and are independent of the main
results above. Each requires its own set of experiment directories (produced
by re-running `run_experiment.py` with the relevant config overrides); fill in
the `'TODO: env_name run'` placeholders accordingly. The labels are kept to
indicate what each run should be.

In [ ]:
# Ablation: amount of initial (pre-deployment) training.
plot_multirun([
    RunInfo('TODO: env_name run', '0', orange),
    RunInfo('TODO: env_name run', '100', yellow),
    RunInfo('TODO: env_name run', '200', purple),
    RunInfo('TODO: env_name run', '300', pink),
    RunInfo('TODO: env_name run', '1000', green)
    ],
    name='initial_training_steps',
    conf=PlotConfig(
        show_y_label=True,
        show_x_label=True,
        show_divider=False,
        smoothing=1,
        show_legend=True,
        legend_ncols=2,
        legend_offset_x=0.3,
    )
)
# Ablation: tampering-handling strategy.
plot_multirun([
    RunInfo('TODO: env_name run', 'Check all', orange),
    RunInfo('TODO: env_name run', 'Check by reward', yellow),
    RunInfo('TODO: env_name run', 'Discard by reward', purple),
    RunInfo('TODO: env_name run', 'Each Step', pink),
    RunInfo('TODO: env_name run', 'Punishment', green),
    ],
    name='ablations',
    conf=PlotConfig(
        show_y_label=True,
        show_x_label=True,
        divider_label='',
        show_legend=True,
        legend_ncols=2,
    )
)

In [ ]:
# Ablation: amount of deployment training before tampering becomes possible.
plot_multirun([
    RunInfo('TODO: env_name run', '100', purple),
    RunInfo('TODO: env_name run', '300', pink),
    RunInfo('TODO: env_name run', '500', green, deployment_label='0', deployment_color=orange),
    ],
    name='tampering_training_steps',
    conf=PlotConfig(
        show_y_label=True,
        show_legend=True,
        legend_ncols=3
    )
)
plot_run('TODO: env_name run', PlotConfig(
    show_y_label=True,
    smoothing=1,
    x_min=0,
    y_min=0,
    y_max=300,
    y_ticks=np.arange(0, 300, 50),
    override_with_env_defaults=False,
    show_legend=True,
    divider_label='$\\mathit{No\\ Inconsistency}$',
    legend_ncols=2,
    legend_columnspacing=-1,
))
# Ablation: number of supervisors (Absent Supervisor env).
plot_multirun([
    RunInfo('TODO: env_name run', '1 supervisor', green),
    RunInfo('TODO: env_name run', '10 supervisors', orange),
], PlotConfig(
    show_legend=True,
    legend_ncols=2,
    show_x_label=True,
),
'num_supervisors',
)
# Ablation: deployment without walls (Absent Supervisor env).
plot_run('TODO: env_name run', PlotConfig(
    show_legend=True,
    legend_ncols=2,
    show_x_label=True,
))

In [ ]:
# Appendix: latent reward-model variant.
plot_run('TODO: env_name run', PlotConfig(
    show_legend=True,
    legend_ncols=2,
    show_x_label=True,
))